# Imports & Data Loading

In [ ]:
!pip install nltk
import nltk
import re
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

import gc
import os

In [ ]:
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Bidirectional, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Data Load**

In [ ]:
#training data
train_url = 'https://drive.google.com/file/d/1FaHdT2UOnPgT7Enl-EYxrqLM8vwfBKrh/view'
id = train_url.split("/")[-2]
new_link = f'https://drive.google.com/uc?id={id}'
train_df = pd.read_csv(new_link)
display(train_df.head())

,News Headline,News Topic
0,<html> <body> News Headlines:\n <br> <b> Presi...,Business
1,<html> <body> News Headlines:\n <br> <b> Will ...,Science and Technology
2,<html> <body> News Headlines:\n <br> <b> Updat...,Business
3,<html> <body> News Headlines:\n <br> <b> Workp...,Business
4,<html> <body> News Headlines:\n <br> <b> Fish ...,Sports


# Encoding & Splitting Dataset

In [ ]:
#Label Encoding and split test and train dataset
X_train, y_train = train_df['News Headline'], train_df['News Topic']
y_train = y_train.map({'Business': 0, 'Science and Technology': 1,'Sports':2, 'World News': 3})

In [ ]:
#Split for validation

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [ ]:
# One-hot encode the labels for the DNN, RNN models
y_train_oh = to_categorical(y_train, num_classes=4)
y_val_oh = to_categorical(y_val, num_classes=4)

# Data Preprocessing (extreme)

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [ ]:
def extreme_preprocess(text):
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # remove dataset-specific phrase
    text = text.replace("news headlines:", "")

    # remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # remove short words
    words = [w for w in words if len(w) > 2]

    # stemming (aggressive)
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

In [ ]:
#extreme Preprocessing for all models
X_train_extreme = X_train.apply(extreme_preprocess)
X_val_extreme = X_val.apply(extreme_preprocess)

del X_train
del X_val
del y_train
del y_val
gc.collect()

NameError: name 'X_train' is not defined

# Tfidf Vectorization

In [ ]:
#Tfidf Vectorization for LR model and DNN
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf_extreme = tfidf_vectorizer.fit_transform(X_train_extreme)
X_val_tfidf_extreme = tfidf_vectorizer.transform(X_val_extreme)

# Skipgram

In [ ]:
!pip install gensim
import gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.4 MB/s eta 0:00:00


In [ ]:
# skipgram model training
sentences = [sentence.lower().split() for sentence in X_train_extreme]

sg_model_extreme = Word2Vec(vector_size=100, window=5, sg=1, min_count=1)

sg_model_extreme.build_vocab(sentences)
sg_model_extreme.train(sentences, total_examples=len(sentences), epochs=10)

(18173405, 18423520)

In [ ]:
# word to vector conversion
import numpy as np

def sentence_to_vectors(text):
    words = text.lower().split()
    vectors = []

    for word in words:
        if word in sg_model_extreme.wv:
            vectors.append(sg_model_extreme.wv[word])
        else:
            vectors.append(np.zeros(sg_model_extreme.vector_size))
    return np.array(vectors)


X_train_vec_extreme = [sentence_to_vectors(s) for s in X_train_extreme]
X_val_vec_extreme = [sentence_to_vectors(s) for s in X_val_extreme]

In [ ]:
# padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = 50

X_train_vec_extreme = pad_sequences(X_train_vec_extreme, maxlen=max_len, padding='post', dtype='float32')
X_val_vec_extreme = pad_sequences(X_val_vec_extreme, maxlen=max_len, padding='post', dtype='float32')

In [ ]:
del X_train_extreme
del X_val_extreme

del sentences
del sg_model_extreme
gc.collect()

0

# **Model Save and  Reuse**

In [ ]:
save_path = "/content/drive/MyDrive/extreme_preprocessing"

import os
os.makedirs(save_path, exist_ok=True)

from scipy import sparse
import numpy as np

# TF-IDF
sparse.save_npz(f"{save_path}/X_train_tfidf_extreme.npz", X_train_tfidf_extreme)
sparse.save_npz(f"{save_path}/X_val_tfidf_extreme.npz", X_val_tfidf_extreme)

# Skipgram
np.save(f"{save_path}/X_train_vec_extreme.npy", X_train_vec_extreme)
np.save(f"{save_path}/X_val_vec_extreme.npy", X_val_vec_extreme)

# Labels
np.save(f"{save_path}/y_train_oh.npy", y_train_oh)
np.save(f"{save_path}/y_val_oh.npy", y_val_oh)

print("✅ Saved to Google Drive!")

✅ Saved to Google Drive!


In [ ]:
save_path = "/content/drive/MyDrive/extreme_preprocessing"

from scipy import sparse
import numpy as np

X_train_tfidf_extreme = sparse.load_npz(f"{save_path}/X_train_tfidf_extreme.npz")
X_val_tfidf_extreme   = sparse.load_npz(f"{save_path}/X_val_tfidf_extreme.npz")

X_train_vec_extreme = np.load(f"{save_path}/X_train_vec_extreme.npy")
X_val_vec_extreme   = np.load(f"{save_path}/X_val_vec_extreme.npy")

y_train_oh = np.load(f"{save_path}/y_train_oh.npy")
y_val_oh   = np.load(f"{save_path}/y_val_oh.npy")

In [ ]:
max_len = X_train_vec_extreme.shape[1]
embedding_dim = X_train_vec_extreme.shape[2]

# DNN

In [ ]:
def dnn_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):
    dnn_model_extreme = Sequential([
        Dense(units, activation="relu", input_shape=(X_train_tfidf_extreme.shape[1],)),
        Dropout(dropout),
        Dense(dense_units, activation="relu"),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(4, activation="softmax")
    ])
    dnn_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"])
    return dnn_model_extreme

# SimpleRNN

In [ ]:
def simple_rnn_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    simple_rnn_model_extreme = Sequential([
        SimpleRNN(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    simple_rnn_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return simple_rnn_model_extreme

# LSTM

In [ ]:
def lstm_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    lstm_model_extreme = Sequential([
        LSTM(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    lstm_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return lstm_model_extreme

# GRU

In [ ]:
def gru_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    gru_model_extreme = Sequential([
        GRU(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    gru_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return gru_model_extreme

# Bidirectional SimpleRNN

In [ ]:
def bidirectional_simple_rnn_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_simple_rnn_model_extreme = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(SimpleRNN(units, input_shape=(max_len, embedding_dim))),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_simple_rnn_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return bidirectional_simple_rnn_model_extreme

# Bidirectional LSTM

In [ ]:
def bidirectional_lstm_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_lstm_model_extreme = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(LSTM(units)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_lstm_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return bidirectional_lstm_model_extreme

# Bidirectional GRU

In [ ]:
def bidirectional_gru_extreme(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_gru_model_extreme = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(GRU(units)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_gru_model_extreme.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return bidirectional_gru_model_extreme

# **Tuning Part**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def evaluate(model, X, y_oh):
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
    y_true = np.argmax(y_oh, axis=1)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    return acc, f1

In [ ]:
def run_experiment(model_fn, config, X_train, y_train, X_val, y_val, branch, model_name, run_id):

    model = model_fn(
        units=config["units"],
        dropout=config["dropout"],
        dense_units=config["dense_units"],
        lr=config["lr"]
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    val_acc, val_f1 = evaluate(model, X_val, y_val)
    train_acc, train_f1 = evaluate(model, X_train, y_train)

    res =  {
        "run_id": run_id,
        "branch": branch,
        "model": model_name,
        "units": config["units"],
        "dropout": config["dropout"],
        "dense_units": config["dense_units"],
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    }

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return res

In [ ]:
models = [
    {
        "name": "DNN",
        "fn": dnn_extreme,
        "X_train": X_train_tfidf_extreme,
        "X_val": X_val_tfidf_extreme
    },
    {
        "name": "SimpleRNN",
        "fn": simple_rnn_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    },
    {
        "name": "LSTM",
        "fn": lstm_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    },
    {
        "name": "GRU",
        "fn": gru_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    },
    {
        "name": "Bi_SimpleRNN",
        "fn": bidirectional_simple_rnn_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    },
    {
        "name": "Bi_LSTM",
        "fn": bidirectional_lstm_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    },
    {
        "name": "Bi_GRU",
        "fn": bidirectional_gru_extreme,
        "X_train": X_train_vec_extreme,
        "X_val": X_val_vec_extreme
    }
]

In [ ]:
configs = [
    {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.001, "batch_size": 64, "epochs": 6},
    {"units": 128,"dropout": 0.50,"dense_units": 64,"lr": 0.001,"batch_size": 32,"epochs": 8},
    {"units": 128,"dropout": 0.20,"dense_units": 128,"lr": 0.001,"batch_size": 32,"epochs": 10},
    {"units": 32,"dropout": 0.60,"dense_units": 32,"lr": 0.0005,"batch_size": 128,"epochs": 4},
    {"units": 64,"dropout": 0.40,"dense_units": 128,"lr": 0.005,"batch_size": 64,"epochs": 5}
]

model_configs = {
    "DNN": [
        {"units": 32, "dropout": 0.55, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
        {"units": 48, "dropout": 0.50, "dense_units": 64, "lr": 0.0005, "batch_size": 128, "epochs": 5},
        {"units": 64, "dropout": 0.45, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
    ],

    "SimpleRNN": [
        {"units": 48, "dropout": 0.35, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 96, "dropout": 0.40, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
    ],

    "LSTM": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "GRU": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "Bi_SimpleRNN": [
        {"units": 48, "dropout": 0.35, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 96, "dropout": 0.40, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
    ],

    "Bi_LSTM": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "Bi_GRU": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],
}

In [ ]:
run_id = 54

In [ ]:
file_path = "/content/drive/MyDrive/extreme_preprocessing/extreme_tuning_results.csv"
for m in range(6, 7):
    model_info = models[m]
    model_name = model_info["name"]
    print(f"\n🔹 Tuning {model_name}...")

    for i in range(3):
        cfg = model_configs[model_name][i]

        res = run_experiment(
            model_fn=model_info["fn"],
            config=cfg,
            X_train=model_info["X_train"],
            y_train=y_train_oh,
            X_val=model_info["X_val"],
            y_val=y_val_oh,
            branch="Extreme",
            model_name = model_info["name"],
            run_id=run_id
        )

        new_df = pd.DataFrame([res])
        new_df.to_csv(
            file_path,
            mode='a',
            header=not os.path.exists(file_path),
            index=False
        )

        print(f"Saved run {run_id}")

        run_id += 1


🔹 Tuning Bi_GRU...
Epoch 1/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 186s 76ms/step - accuracy: 0.8940 - loss: 0.3074 - val_accuracy: 0.9030 - val_loss: 0.2820
Epoch 2/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 197s 74ms/step - accuracy: 0.9101 - loss: 0.2566 - val_accuracy: 0.9080 - val_loss: 0.2725
Epoch 3/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 161s 68ms/step - accuracy: 0.9179 - loss: 0.2321 - val_accuracy: 0.9206 - val_loss: 0.2346
Epoch 4/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 202s 68ms/step - accuracy: 0.9253 - loss: 0.2107 - val_accuracy: 0.9238 - val_loss: 0.2225
Epoch 5/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 154s 64ms/step - accuracy: 0.9332 - loss: 0.1930 - val_accuracy: 0.9194 - val_loss: 0.2365
Epoch 6/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 204s 65ms/step - accuracy: 0.9381 - loss: 0.1748 - val_accuracy: 0.9281 - val_loss: 0.2181
Epoch 7/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 154s 65ms/step - accuracy: 0.9444 - loss: 0.1562 - val_accuracy: 0.9273 - val_loss: 0.2235
Epoch 8/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 203s 65